# CS 195: Natural Language Processing
## Retrieval-Augmented Generation (RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F5_1_RAGCourseInfo.ipynb)


## References

Hugging Face Chat Basics: https://huggingface.co/docs/transformers/en/conversations

Sentence-Transformers documentation: https://www.sbert.net/

Chapter 11: Information Retrieval and Retrieval-Augmented Generation: https://web.stanford.edu/~jurafsky/slp3/11.pdf


In [1]:
#import sys
#!{sys.executable} -m pip install transformers sentence-transformers accelerate requests

In [2]:
# word wrap for jupyter notebook - probably important with this notebook since we will be printing out a lot of text
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

## Retrieval-Augmented Generation (RAG)

Last time, we talked about the basic idea behind RAG, and today we'll try it out with a small model.

As a reminder, there are three main steps:
1. **Retrieve** documents relevant to the user's question
2. **Augment** the prompt with those documents
3. **Generate** an answer using a chat model

The **retrieve** step is often done using a semantic search like we did [last time](https://github.com/ericmanley/S26-CS195NLP/blob/main/F4_3_SemanticSearchEmbeddings.ipynb). This notebook shows a possible solution to the Applied Exploration from last time where we'll do semantic search  with the [Drake course information documents](https://github.com/ericmanley/S26-CS195NLP/blob/main/data/f25_course_information.json)

For prompting and generating responses with a chat model, we'll follow what we did in [F2_1_ChatInstruct.ipynb](https://github.com/ericmanley/S26-CS195NLP/blob/main/F2_1_ChatInstruct.ipynb)

Possible goal: build something like [Professor Griff](https://ericmanley.github.io/professor_griff.html)

## Load the Course Information Data

We'll use the same course information JSON file from the semantic search notes.


In [3]:
# load directly from github using the requests library
import requests
import json

response = requests.get("https://raw.githubusercontent.com/ericmanley/S26-CS195NLP/refs/heads/main/data/f25_course_information.json")
data = json.loads(response.text)

#let's just print the first record to remember what they look like
print(data[0])


{'id': 213644, 'term': 'Fall 2025', 'course_number': 'ACCT 041', 'subject': 'Accounting', 'title': 'INTRODUCTION TO FINANCIAL ACCOUNTING', 'course_search_url': 'https://catalog.drake.edu/course-search/?details&srcdb=2024&code=ACCT%20041', 'prereq': 'Prerequisite(s): None', 'description': '\n\n    The elements of the financial statements, accounting for deferrals, the double-entry accounting system, internal control and cash, receivables and payables, inventory, operational assets, long-term debt, equity transactions, income measurement, and comprehensive treatment of the balance sheet, the income statement and the statement of cash flows.  Financial statement analysis will be integrated throughout the course.\n    \n\n', 'credit_hours': None, 'faculty': ['Joyce Njoroge'], 'attributes': ['Critical Thinking'], 'location': ['ALIB 0010'], 'times': ['Monday, Wednesday 0930-1045'], 'filename': 'course_213644.json'}


## Turn Each Course into a Searchable Text Chunk

The HuggingFace `text-generation` pipeline cannot take the raw json as input, so we first need to convert them into some kind of textual form instead.


In [4]:
def course_record_to_text(record):
    faculty = ', '.join(record.get('faculty') or [])
    attributes = ', '.join(record.get('attributes') or [])
    location = ', '.join(record.get('location') or [])
    times = ', '.join(record.get('times') or [])

    parts = [
        f"Course: {record.get('course_number', '')}",
        f"Subject: {record.get('subject', '')}",
        f"Title: {record.get('title', '')}",
        f"Description: {record.get('description', '').strip()}",
        f"Prerequisites: {record.get('prereq', '')}",
        f"Faculty: {faculty}",
        f"Attributes: {attributes}",
        f"Location: {location}",
        f"Times: {times}",
    ]
    return "\n".join(parts)

course_texts = [course_record_to_text(record) for record in data]
print(course_texts[0])


Course: ACCT 041
Subject: Accounting
Title: INTRODUCTION TO FINANCIAL ACCOUNTING
Description: The elements of the financial statements, accounting for deferrals, the double-entry accounting system, internal control and cash, receivables and payables, inventory, operational assets, long-term debt, equity transactions, income measurement, and comprehensive treatment of the balance sheet, the income statement and the statement of cash flows.  Financial statement analysis will be integrated throughout the course.
Prerequisites: Prerequisite(s): None
Faculty: Joyce Njoroge
Attributes: Critical Thinking
Location: ALIB 0010
Times: Monday, Wednesday 0930-1045


## Build the Retrieval Index

We'll use a pretrained sentence-transformer to create one embedding vector per course


In [5]:
from sentence_transformers import SentenceTransformer
import torch
import torch.nn.functional as F

retrieval_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
course_embeddings = retrieval_model.encode(course_texts, convert_to_tensor=True)

print('embedding matrix shape:', course_embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding matrix shape: torch.Size([1175, 384])


## Retrieval Function

We'll take the code we worked on last time for scoring each document based on semantic similarity and put it into a function.

This version adds a few things
* we'll keep a list of all the semantic similarity scores instead of printing them out
* we'll use `torch.topk` to find the top 3 related documents to the query
* we'll return a list of records containing both the scores and the documents


In [6]:
def cosine(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

def retrieve_documents(query, documents, doc_embeddings, top_k=3):
    query_embedding = retrieval_model.encode( query, convert_to_tensor=True)

    scores = [] # keep track of all scores in a list

    for d_idx in range(len(documents)):
        doc_sim_score = cosine(query_embedding, doc_embeddings[d_idx])
        scores.append(doc_sim_score)
        #print(doc_sim_score.item(), documents[d_idx])

    # convert the scores list to a tensor, find the top k, return the indices and then convert back to a list
    top_idx = torch.topk(torch.tensor(scores), k=top_k).indices.tolist()

    results = []
    for idx in top_idx:
        results.append({
            'score': scores[idx].item(),
            'document': documents[idx]
        })
    return results


## Test Retrieval by Itself First

Before we involve a chat model, we should make sure retrieval is bringing back good course records.


In [7]:
test_query = 'Which courses are about artificial intelligence?'
retrieved = retrieve_documents(test_query, course_texts, course_embeddings, top_k=3)

print('QUERY:', test_query)
for i, hit in enumerate(retrieved, start=1):
    print()
    print(f'HIT {i} | score={hit["score"]:.3f}')
    print(hit['document'])


QUERY: Which courses are about artificial intelligence?

HIT 1 | score=0.694
Course: AI 010
Subject: Artificial Intelligence
Title: INTERDISCIPLINARY PERSPECTIVES ON ARTIFICIAL INTELLIGENCE
Description: This course serves as an introduction to the Artificial Intelligence major and minor. The aim of the course is to provide an
overview of Artificial Intelligence through the lens of multiple disciplines, including computer science, philosophy,
psychology, linguistics, literature, and business. The course will feature a number of outside speakers with expertise
drawn from the above-listed areas. Upon completion of the course, students should have a foundation of the main ideas
and concepts they will explore more deeply in later classes in the program.
Prerequisites: Prerequisite(s): None
Faculty: Christopher Porter
Attributes: 
Location: SCB 0201
Times: Monday, Wednesday 1400-1515

HIT 2 | score=0.528
Course: PHIL 113
Subject: Philosophy
Title: AI ETHICS
Description: An examination of rec

## Load a Small Chat Model

Let's see how this works with the [SmolLM3-3B](https://huggingface.co/HuggingFaceTB/SmolLM3-3B) model
* `device_map='auto'` is something I learned can automatically detect CUDA/MPS so you don't have to pass it in manually like we did before

In [8]:
from transformers import pipeline

MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'

chatbot = pipeline('text-generation', model=MODEL_NAME, dtype='auto', device_map='auto')


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## Ask the Model Without Retrieval - for comparison

First, let's see what happens when we ask the model for a response but don't include any of the retrieved documents.


In [9]:
question = 'Which professors teach courses about artificial intelligence, and what do those courses cover?'

In [10]:
no_rag_messages = [
    {'role': 'system', 'content': 'You are a helpful course assistant.'},
    {'role': 'user', 'content': question},
]

no_rag_response = chatbot(no_rag_messages)

print('WITHOUT RAG:')
print(no_rag_response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITHOUT RAG:
<think>
Okay, the user is asking about professors who teach courses on artificial intelligence and what those courses cover. Let me start by recalling some well-known AI professors. There's Fei-Fei Li, who's a prominent figure in AI and computer vision. Then there's Yoshua Bengio, who's a big name in deep learning. Andrew Ng is another one, known for his work at Stanford and Coursera courses. 

Wait, I should make sure I'm not missing any other key names. Maybe Stuart Russell and Peter Norvig from UC Berkeley? They co-authored the AI textbook. Also, there's Demis Hassabis, the founder of DeepMind, though he's more of an entrepreneur now. 

Now, for the courses they teach, I need to outline the main topics. AI typically covers AI basics, machine learning, neural networks, natural language processing, robotics, and ethics. Each professor might have their own specialization, so the courses might differ. For example, Fei-Fei Li's courses might focus more on computer vision and

## Build Model Instructions Using Retrieved Documents

The prompt should tell the model:
- what the user's question is
- what retrieved course information it may use
- what to do if the retrieved context is not enough

This is one of the most important parts of a RAG system.

In [11]:
# retrieve relevant documents
retrieved = retrieve_documents(question, course_texts, course_embeddings, top_k=3)

# create the instructions for the chatbot
rag_messages = [
    {
        'role': 'system',
        'content': (
            'You are a helpful course assistant. '
            'Answer using the retrieved course information when possible. '
            'If the answer is not supported by the retrieved context, say that you do not know based on the provided course information.'
        ),
    },
    {
        'role': 'user',
        'content': (
            f"Question: {question}\n\n"
            f"Retrieved course information:\n{retrieved}\n\n"
            'Please answer the question using the retrieved course information.'
        ),
    },
]

# prompt the model with the RAG context included
rag_response = chatbot(rag_messages)


print('WITH RAG:')
print(rag_response[0]['generated_text'][-1]['content'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITH RAG:
<think>

</think>
Based on the retrieved course information, the following professors teach courses related to Artificial Intelligence:

1. Christopher Porter teaches the course titled "INTERDISCIPLINARY PERSPECTIVES ON ARTIFICIAL INTELLIGENCE" (Course: AI 010). The course covers an overview of Artificial Intelligence through the lens of multiple disciplines, including computer science, philosophy, psychology, linguistics, and business. 

2. Jennifer McCrickerd teaches the course titled "AI ETHICS" (Course: PHIL 113). The course examines recent discussions of ethical issues in AI, including issues of privacy, bias, problematic influence, and social consequences of AI.

The other course retrieved, "CS 065 - Introduction to Computer Science I" (Course: CS 065), is not directly related to Artificial Intelligence but covers foundational topics in computer science such as algorithms, programming, and data presentation.

Please note that the provided course information does not cov

In [12]:
print('The entire chat template for review:')
print(rag_response)

The entire chat template for review:
[{'generated_text': [{'role': 'system', 'content': 'You are a helpful course assistant. Answer using the retrieved course information when possible. If the answer is not supported by the retrieved context, say that you do not know based on the provided course information.'}, {'role': 'user', 'content': "Question: Which professors teach courses about artificial intelligence, and what do those courses cover?\n\nRetrieved course information:\n[{'score': 0.6303054094314575, 'document': 'Course: AI 010\\nSubject: Artificial Intelligence\\nTitle: INTERDISCIPLINARY PERSPECTIVES ON ARTIFICIAL INTELLIGENCE\\nDescription: This course serves as an introduction to the Artificial Intelligence major and minor. The aim of the course is to provide an\\noverview of Artificial Intelligence through the lens of multiple disciplines, including computer science, philosophy,\\npsychology, linguistics, literature, and business. The course will feature a number of outside s

## Group Discussion

What are some deficiencies in the workflow?

List some ideas of things you'd like to try to improve it.

## Applied Exploration

Perform an evaluation of at least three models.

I suggest either using different sized models within one family. For example
* HuggingFaceTB/SmolLM2-135M-Instruct
* HuggingFaceTB/SmolLM2-360M-Instruct
* HuggingFaceTB/SmolLM2-1.7B-Instruct

Or, three different models of similar sizes and release dates
* HuggingFaceTB/SmolLM3-3B
* Qwen/Qwen3-4B
* mistralai/Ministral-3-3B-Instruct-2512

*Note:* There are QWen 3.5 models available (e.g., https://huggingface.co/Qwen/Qwen3.5-0.8B ), but they use a `image-text-to-text` pipeline, and I haven't experimented with it. Feel free to try it out and report back.

Feel free to make whatever changes you think are appropriate (edit the RAG prompt, change the number of entries you return, etc.)

Benchmark: Come up with 5 different questions about the course schedule data to use

For each model, evaluate its performance using your benchmark and summarize your findings.


In [13]:
from transformers import pipeline
import pandas as pd
import torch

# -----------------------------
# 1. Benchmark Questions
# -----------------------------

benchmark_questions = [
    "Which courses are about artificial intelligence, and who teaches them?",
    "Which computer science courses have prerequisites?",
    "Which courses meet on Monday, Wednesday, and Friday?",
    "Which courses are taught by Eric Manley?",
    "Which courses include topics related to data, machine learning, or algorithms?"
]

# -----------------------------
# 2. Models to Evaluate
# -----------------------------

models_to_test = [
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct"
]

# -----------------------------
# 3. RAG Prompt Function
# -----------------------------

def build_rag_messages(question, retrieved_docs):
    context = "\n\n".join([
        f"Document {i+1}:\n{doc['document']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    return [
        {
            "role": "system",
            "content": (
                "You are a helpful course schedule assistant. "
                "Answer ONLY using the retrieved course information. "
                "If the retrieved information does not contain the answer, "
                "say you do not know based on the provided course information. "
                "Be clear, concise, and include course numbers, titles, and faculty when available."
            )
        },
        {
            "role": "user",
            "content": (
                f"Question: {question}\n\n"
                f"Retrieved course information:\n{context}\n\n"
                "Answer the question using only the retrieved course information."
            )
        }
    ]

# -----------------------------
# 4. Run Evaluation
# -----------------------------

results = []

for model_name in models_to_test:
    print("=" * 80)
    print("Loading model:", model_name)
    print("=" * 80)

    chatbot = pipeline(
        "text-generation",
        model=model_name,
        dtype="auto",
        device_map="auto"
    )

    for question in benchmark_questions:
        print("\nQUESTION:", question)

        retrieved = retrieve_documents(
            question,
            course_texts,
            course_embeddings,
            top_k=5
        )

        rag_messages = build_rag_messages(question, retrieved)

        response = chatbot(
            rag_messages,
            max_new_tokens=300,
            do_sample=False
        )

        answer = response[0]["generated_text"][-1]["content"]

        print("\nANSWER:")
        print(answer)

        results.append({
            "model": model_name,
            "question": question,
            "retrieved_documents": retrieved,
            "answer": answer
        })

# -----------------------------
# 5. Save Results
# -----------------------------

results_df = pd.DataFrame(results)
results_df.to_csv("rag_model_evaluation_results.csv", index=False)

results_df

Loading model: HuggingFaceTB/SmolLM2-135M-Instruct


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
Based on the retrieved course information, EDUC 299, DOC 350, EDUC 263, and EDCR 203 all meet on Monday, Wednesday, and Friday.

QUESTION: Which courses are taught by Eric Manley?


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
Based on the retrieved course information, Eric Manley teaches the following courses:

1. EDL 292 - Educational Leadership
2. EDUC 266 - Student Teaching - Elementary
3. EDCR 206 - Instru Ldrshp & Teaching Pract
4. EDUC 263 - Field Placement Capstone Seminar
5. EDUC 208 - Student Development and Learning Theory

QUESTION: Which courses include topics related to data, machine learning, or algorithms?

ANSWER:
Based on the retrieved course information, the following courses include topics related to data, machine learning, or algorithms:

1. CS 167: MACHINE LEARNING
2. CS 065: INTRODUCTION TO COMPUTER SCIENCE I
3. CS 066: INTRODUCTION TO COMPUTER SCIENCE II
4. CS 167: MACHINE LEARNING

All of these courses are related to computer science and programming, and they cover topics such as algorithms, data structures, and machine learning.


,model,question,retrieved_documents,answer
0,HuggingFaceTB/SmolLM2-135M-Instruct,Which courses are about artificial intelligenc...,"[{'score': 0.6731440424919128, 'document': 'Co...",Question: Which courses are about artificial i...
1,HuggingFaceTB/SmolLM2-135M-Instruct,Which computer science courses have prerequisi...,"[{'score': 0.7252922654151917, 'document': 'Co...",Document 1:\nCourse: CS 065\nSubject: Computer...
2,HuggingFaceTB/SmolLM2-135M-Instruct,"Which courses meet on Monday, Wednesday, and F...","[{'score': 0.5277459621429443, 'document': 'Co...","Question: Which courses meet on Monday, Wednes..."
3,HuggingFaceTB/SmolLM2-135M-Instruct,Which courses are taught by Eric Manley?,"[{'score': 0.5654405355453491, 'document': 'Co...",Question: Which courses are taught by Eric Man...
4,HuggingFaceTB/SmolLM2-135M-Instruct,"Which courses include topics related to data, ...","[{'score': 0.6587942838668823, 'document': 'Co...",Question: Which courses include topics related...
5,HuggingFaceTB/SmolLM2-360M-Instruct,Which courses are about artificial intelligenc...,"[{'score': 0.6731440424919128, 'document': 'Co...",Course: AI 010\n\nWho teaches the course about...
6,HuggingFaceTB/SmolLM2-360M-Instruct,Which computer science courses have prerequisi...,"[{'score': 0.7252922654151917, 'document': 'Co...",CS 065 has prerequisites.
7,HuggingFaceTB/SmolLM2-360M-Instruct,"Which courses meet on Monday, Wednesday, and F...","[{'score': 0.5277459621429443, 'document': 'Co...",Course: EDCR 203\nSubject: Instru Ldrshp &amp;...
8,HuggingFaceTB/SmolLM2-360M-Instruct,Which courses are taught by Eric Manley?,"[{'score': 0.5654405355453491, 'document': 'Co...","Eric Manley taught courses EDUC 266, EDCR 206,..."
9,HuggingFaceTB/SmolLM2-360M-Instruct,"Which courses include topics related to data, ...","[{'score': 0.6587942838668823, 'document': 'Co...",CS 167


The Applied Exploration tested three instruction-tuned language models within the Retrieval-Augmented Generation (RAG) pipeline using the course schedule dataset. The models included HuggingFaceTB/SmolLM2-135M-Instruct, HuggingFaceTB/SmolLM2-360M-Instruct, and HuggingFaceTB/SmolLM2-1.7B-Instruct. Each model was evaluated using the same benchmark questions related to artificial intelligence, prerequisites, instructors, meeting times, and machine learning topics. The smaller models were able to answer some questions but often produced incomplete or repetitive responses. The larger 1.7B model generated more detailed and accurate answers by using the retrieved course information more effectively. The results showed that larger language models generally perform better in RAG systems, but the quality of the retrieval step is still very important because the generated answers depend heavily on the relevance of the retrieved documents.

## Small Project Prototype idea (Creative Synthesis)

Build a RAG-based application that allows students to upload their notes from a class and ask the chatbot questions about them.

## Large Project Prototype idea (Creative Synthesis)

Build a RAG-based application that allows an instructor to upload video recordings of their class and plays video clips in response to student questions.

In [14]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import whisper

model = whisper.load_model("base")  # tiny, base, small, medium, large

result = model.transcribe("videoplayback.mp4")

for segment in result["segments"]:
    start = segment["start"]
    end = segment["end"]
    text = segment["text"]
    print(f"[{start:.2f} - {end:.2f}] {text}")

## Extended Implementation idea (Creative Synthesis)

Swap out our similarity search code with a vector database like Chroma: https://docs.trychroma.com/docs/overview/getting-started
* This is probably easier to do than to implement it from scratch like we did
* You should try out the [persistent](https://docs.trychroma.com/docs/run-chroma/clients#persistent-client) and/or [client-server](https://docs.trychroma.com/docs/run-chroma/client-server) modes